In [ ]:
import os
import numpy as np

# 建立 hex 輸出資料夾（配合硬體 tb 的相對路徑）
os.makedirs("hex", exist_ok=True)

# =====================================================================
# 1. 基礎參數設定 (與硬體 PPU_TB / ASIC.svh 一致)
# =====================================================================
TOKEN_TILE = 16
CHANNEL_TILE = 16
TILE_ELEMS = TOKEN_TILE * CHANNEL_TILE  # 256 個元素
ZERO_POINT = 128

# 模擬 GELU LUT (256字，對應 GELU_Unit.sv 中的硬體初始值)
# 硬體中使用索引 0~255，此處建立與硬體代碼完全相同的映射
GELU_ROM = np.zeros(256, dtype=np.uint8)
# 根據 GELU_Unit.sv 前幾行範例與對稱性，此處提供標準 ViT 定點數 GELU 模擬邏輯
# 實際上硬體會讀取 data_in[15:8] 作為無符號 0~255 索引
for idx in range(256):
    # 將硬體索引轉回 8-bit 有符號數進行近似模擬（符合硬體註解的 signed 映射）
    val_signed = idx if idx < 128 else idx - 256
    x = val_signed * 0.256
    if x < -4:
        gelu_val = 0
    elif x > 4:
        gelu_val = x
    else:
        # 標準 GELU 函數
        gelu_val = x * 0.5 * (1.0 + np.tanh(np.sqrt(2.0 / np.pi) * (x + 0.044715 * (x**3))))
    # 量化回 uint8 (以 128 為零點)
    q_gelu = int(np.clip(gelu_val / 0.01 + 128, 0, 255))
    GELU_ROM[idx] = q_gelu

# =====================================================================
# 2. PPU 核心硬體行為的 Python 模擬函數
# =====================================================================

def hardware_requant(data_in_int32, scaling_factor):
    """ 模擬 Requant_Unit.sv """
    # 1. 算術右移 (Arithmetic Right Shift)
    # Python 對有符號整數做 >> 即為算術右移
    shifted = data_in_int32 >> scaling_factor
    
    # 2. 正負溢位偵測與飽和截斷 (以 8-bit signed [-128, 127] 為硬體切分基準)
    # 模擬硬體：如果大於 127 則上溢出，小於 -128 則下溢出
    clipped = np.clip(shifted, -128, 127)
    
    # 3. 轉換為以 128 為零點的 uint8 格式
    data_out = (clipped + ZERO_POINT).astype(np.uint8)
    return data_out

def hardware_residual_add(main_q, residual_q):
    """ 模擬 Residual_Add_Unit.sv """
    # q_out = clamp(q_main + q_residual - 128, 0, 255)
    sum_q = main_q.astype(np.int16) + residual_q.astype(np.int16) - ZERO_POINT
    data_out = np.clip(sum_q, 0, 255).astype(np.uint8)
    return data_out

def save_to_hex(filename, data_array, is_int32=False):
    """ 將 1D 陣列儲存成 Verilog $readmemh 格式的文字檔 (修正負數補數與寬度問題) """
    filepath = os.path.join("hex", filename)
    with open(filepath, "w") as f:
        for val in data_array:
            if is_int32:
                # 關鍵：如果是 32-bit 整數，將負數強制轉為 32-bit 無符號補數表示法 (8位十六進位，去除負號)
                val_uint32 = int(val) & 0xFFFFFFFF
                f.write(f"{val_uint32:08X}\n")
            else:
                # 8-bit uint8 資料不變
                f.write(f"{int(val):02X}\n")
    print(f"[成功] 已生成相容 VCS 的測資: {filepath}")

# =====================================================================
# 3. 隨機生成輸入測資 (各 256 個元素)
# =====================================================================
np.random.seed(42)  # 固定隨機種子以確保每次生成解答一致

# psum_in.hex: 模擬來自 Systolic Array 的 32-bit Partial Sum
# 數值刻意涵蓋大範圍以測試 Requant 的算術右移與飽和截斷
psum_in = np.random.randint(-10000, 10000, size=TILE_ELEMS, dtype=np.int32)

# residual_in.hex: 模擬殘差路的 uint8 特徵圖
residual_in = np.random.randint(0, 256, size=TILE_ELEMS, dtype=np.uint8)

# 儲存輸入檔案
save_to_hex("psum_in.hex", psum_in, is_int32=True)
save_to_hex("residual_in.hex", residual_in)

# =====================================================================
# 4. 生成各種 Mode 的 Golden 解答
# =====================================================================

# ---------------------------------------------------------------------
# [Test 1] Mode 00: Attention Output Phase
# 邏輯：Requant(scaling=2) -> Residual Add -> 觸發 RMS
# ---------------------------------------------------------------------
# 1. 執行 Requant (scaling factor = 2)
attn_requant_out = hardware_requant(psum_in, scaling_factor=2)
# 2. 執行 Residual Add (與輸入殘差相加)
golden_attn = hardware_residual_add(attn_requant_out, residual_in)
save_to_hex("golden_attn.hex", golden_attn)

# 計算並印出 Test 1 的第一個 Token (Row 0) 的 RMS 預期統計結果以供比對
# 真實值 = q - 128
centered_attn = golden_attn.reshape(TOKEN_TILE, CHANNEL_TILE).astype(np.int32) - ZERO_POINT
sum_sq_row0_attn = np.sum(centered_attn[0, :] ** 2)
print(f"--> [提示] Test 1 (Attention) Row 0 的單個 Tile 平方和預估值: {sum_sq_row0_attn}")


# ---------------------------------------------------------------------
# [Test 2] Mode 01: FFN FC1 Phase
# 邏輯：GELU 啟用 -> Requant(scaling=0) -> 繞過 Residual -> 無 RMS
# ---------------------------------------------------------------------
# 1. 模擬硬體中進入 GELU Unit 前的查表索引截取 (data_in[15:8])
# 因為 Python 的右移與遮罩對有符號數需要注意，我們將其轉為無符號 8-bit 索引
gelu_lut_indices = ((psum_in >> 8) & 0xFF).astype(np.uint8)
gelu_out_int32 = np.zeros(TILE_ELEMS, dtype=np.int32)

# 從剛剛建立的 GELU_ROM 查表，並還原回 int32 供下一級 Requant 使用
for i in range(TILE_ELEMS):
    # 硬體查表後輸出 uint8，此處將其減去 128 還原為對應的 signed int32 進入 Requant
    gelu_out_int32[i] = int(GELU_ROM[gelu_lut_indices[i]]) - ZERO_POINT

# 2. 執行 Requant (scaling factor = 0)
golden_fc1 = hardware_requant(gelu_out_int32, scaling_factor=0)
save_to_hex("golden_fc1.hex", golden_fc1)


# ---------------------------------------------------------------------
# [Test 3] Mode 10: FFN FC2 Phase (全新加入)
# 邏輯：Requant(scaling=1) -> Residual Add -> 觸發 RMS
# ---------------------------------------------------------------------
# 1. 執行 Requant (scaling factor = 1)
fc2_requant_out = hardware_requant(psum_in, scaling_factor=1)
# 2. 執行 Residual Add (FC2 階段同樣需要與 Residual 殘差相加)
golden_fc2 = hardware_residual_add(fc2_requant_out, residual_in)
save_to_hex("golden_fc2.hex", golden_fc2)

# 計算並印出 Test 3 的第一個 Token (Row 0) 的 RMS 預期統計結果
centered_fc2 = golden_fc2.reshape(TOKEN_TILE, CHANNEL_TILE).astype(np.int32) - ZERO_POINT
sum_sq_row0_fc2 = np.sum(centered_fc2[0, :] ** 2)
print(f"--> [提示] Test 3 (FC2) Row 0 的單個 Tile 平方和預估值: {sum_sq_row0_fc2}")

print("\n[全部完成] 請將生成的 'hex' 資料夾下載並放到你 EDA 伺服器的 DLA/PPU/ 目錄下即可模擬！")